# What is iEEG?

Intracranial electroencephalography (iEEG) provides direct measurements of
neural activity from the human brain at unprecedented spatial and temporal
resolution. This notebook introduces the key concepts and shows what iEEG
signals look like in practice.

:::{note}
This notebook uses **simulated** data so it runs without any downloads.
Later tutorials use real open datasets.
:::

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

## 1. Signal Types

An iEEG electrode records a broadband **Local Field Potential (LFP)**.
Researchers typically band-pass filter this signal into physiologically
meaningful frequency ranges:

| Band | Range | Associated processes |
|------|-------|---------------------|
| Delta | 1–4 Hz | Sleep, deep anesthesia |
| Theta | 4–8 Hz | Memory encoding, navigation |
| Alpha | 8–12 Hz | Idle/inhibition |
| Beta | 13–30 Hz | Motor planning, top-down control |
| Gamma | 30–80 Hz | Local processing |
| **HFA** | **70–150 Hz** | **Proxy for spiking activity** |

## 2. Simulating an iEEG Signal

Let's build a simple synthetic signal to illustrate these bands.

In [ ]:
rng = np.random.default_rng(42)

sfreq  = 1000          # sampling rate (Hz)
dur    = 4.0           # seconds
t      = np.arange(0, dur, 1/sfreq)

# Compose signal from several frequency components
delta  = 2.0  * np.sin(2 * np.pi * 2   * t)          # 2 Hz
theta  = 1.0  * np.sin(2 * np.pi * 6   * t)          # 6 Hz
alpha  = 1.5  * np.sin(2 * np.pi * 10  * t)          # 10 Hz
beta   = 0.5  * np.sin(2 * np.pi * 20  * t)          # 20 Hz
gamma  = 0.3  * np.sin(2 * np.pi * 50  * t)          # 50 Hz
hfa    = 0.2  * np.sin(2 * np.pi * 100 * t)          # 100 Hz
noise  = 0.15 * rng.standard_normal(len(t))

signal = delta + theta + alpha + beta + gamma + hfa + noise
print(f'Signal: {len(t)} samples, {dur:.0f} s at {sfreq} Hz')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)

# Raw signal
axes[0].plot(t, signal, color='#2c3e50', lw=0.6)
axes[0].set_ylabel('Amplitude (a.u.)')
axes[0].set_title('Raw broadband LFP (simulated)')

# Theta band
axes[1].plot(t, theta, color='#2980b9', lw=1)
axes[1].set_ylabel('Amplitude')
axes[1].set_title('Theta band (4–8 Hz)')

# HFA envelope (amplitude modulation)
from scipy.signal import hilbert
hfa_env = np.abs(hilbert(hfa + 0.1 * rng.standard_normal(len(t))))
axes[2].fill_between(t, hfa_env, alpha=0.5, color='#e74c3c')
axes[2].plot(t, hfa_env, color='#c0392b', lw=1)
axes[2].set_ylabel('Amplitude')
axes[2].set_title('High-Frequency Activity envelope (70–150 Hz)')
axes[2].set_xlabel('Time (s)')

plt.tight_layout()
plt.show()

## 3. Power Spectral Density

The **power spectral density (PSD)** shows how signal power is distributed
across frequencies. The characteristic **1/f (pink noise) slope** of neural
signals means low frequencies always dominate — the interesting task-related
signals ride on top of this background.

In [ ]:
from scipy.signal import welch

freqs, psd = welch(signal, fs=sfreq, nperseg=512)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Linear scale
axes[0].plot(freqs, psd, color='#2c3e50', lw=1.2)
axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('Power (a.u.²/Hz)')
axes[0].set_title('PSD — linear scale')
axes[0].set_xlim(0, 200)

# Log-log scale reveals the 1/f slope
axes[1].semilogy(freqs[1:], psd[1:], color='#2980b9', lw=1.2)
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Power (log scale)')
axes[1].set_title('PSD — log scale (shows 1/f structure)')
axes[1].set_xlim(1, 200)

# Annotate bands
band_colors = {'δ': ('#3498db', 2), 'θ': ('#2ecc71', 6),
               'α': ('#e67e22', 10), 'β': ('#9b59b6', 20),
               'γ': ('#e74c3c', 50), 'HFA': ('#c0392b', 100)}
for label, (col, freq) in band_colors.items():
    axes[1].axvline(freq, color=col, lw=0.8, ls='--', alpha=0.7)
    axes[1].text(freq, psd[np.argmin(np.abs(freqs - freq))] * 2,
                 label, fontsize=8, color=col, ha='center')

plt.tight_layout()
plt.show()

## 4. Key Takeaways

- iEEG records a broadband LFP that can be decomposed into frequency bands.
- **High-frequency activity (HFA, 70–150 Hz)** is the most reliable proxy for
  local spiking activity and is widely used as an index of cortical processing.
- The signal has a characteristic **1/f power spectrum** — power decreases
  with increasing frequency.
- In subsequent tutorials we will work with real iEEG data and learn how to
  preprocess, epoch, and analyse it.

:::{tip}
**Next step:** see the [Software & Setup](../content/setup.md) page — install the software and get the data.
:::

## References

- Parvizi & Kastner (2018). Promises and limitations of human intracranial
  electroencephalography. *Nature Neuroscience*, 21, 474–483.
- Ray & Maunsell (2011). Different origins of gamma rhythm and high-gamma
  activity in macaque visual cortex. *PLOS Biology*.